# AAI-540 Feature Store — AWS Starter Notebook

Connects to the feature store through the Glue catalog (`sagemaker_featurestore`) and demonstrates the typical 
offline query pattern.

Splits data and store locally for model training with an optional entry to show how to load into s3 bucket.

**Region:** `us-east-2` · **Workgroup:** `primary` ·
Athena results land at `s3://jonno-lucas-steve-bucket/usd-aai540-group1/athena-results/`.

## Setup

Most SageMaker images already have `awswrangler`, `boto3`, `pandas`,
`numpy`, `sklearn`, `matplotlib`, `seaborn` pre-installed. If you hit
`ImportError`, uncomment the install line below.

In [ ]:
# !pip install -q awswrangler

In [12]:
import boto3
import awswrangler as wr
import pandas as pd

from pathlib import Path


pd.options.display.float_format = "{:,.2f}".format

# Confirm which IAM identity this notebook is running as
sts = boto3.client("sts", region_name="us-east-2")
print(sts.get_caller_identity()["Arn"])

arn:aws:iam::541974874359:user/jonno


In [ ]:
# ---- Constants ----
REGION     = "us-east-2"
BUCKET     = "jonno-lucas-steve-bucket"
PROJECT    = "usd-aai540-group1"
SILVER_DB  = "aai540_silver"
GOLD_DB    = "aai540_gold"
ATHENA_OUT = f"s3://{BUCKET}/{PROJECT}/athena-results/"

# awswrangler picks up region from the boto3 default session
boto3.setup_default_session(region_name=REGION)

## 1. Load the feature store database

The full table is small (~2,500 rows × ~20 columns), so we pull it all
into memory. — Athena charges by data scanned.

In [3]:
df = wr.athena.read_sql_query(
    "SELECT * from x_attendance_y_sales_1779629962",
    database="sagemaker_featurestore",
    s3_output=ATHENA_OUT,
)

print(f"shape:      {df.shape}")
print(f"rows w/ est_attendance:   {(df['total-est-attendance'] > 0).sum()}")
df.head()

shape:      (2554, 7)
rows w/ est_attendance:   829


,record-identifier,event-time,total-est-attendance,taxable-sales-usd,write_time,api_invocation_time,is_deleted
0,06003-2024Q1,2024-03-31T00:00:00Z,0,11326679,2026-05-24 14:26:22.745,2026-05-24 14:21:39,False
1,06055-2022Q1,2022-03-31T00:00:00Z,0,1024327718,2026-05-24 14:26:22.745,2026-05-24 14:21:33,False
2,06043-2022Q1,2022-03-31T00:00:00Z,0,47850811,2026-05-24 14:26:22.745,2026-05-24 14:23:03,False
3,06083-2020Q3,2020-09-30T00:00:00Z,23043,1950327275,2026-05-24 14:25:37.044,2026-05-24 14:21:30,False
4,06087-2020Q3,2020-09-30T00:00:00Z,280,1058749786,2026-05-24 14:25:37.044,2026-05-24 14:22:05,False


In [4]:
# Schema + null check
pd.DataFrame({
    "dtype":    df.dtypes,
    "n_nulls":  df.isna().sum(),
    "n_unique": df.nunique(),
})

,dtype,n_nulls,n_unique
record-identifier,string,0,2552
event-time,string,0,44
total-est-attendance,Int64,0,558
taxable-sales-usd,Int64,0,2552
write_time,datetime64[ns],0,305
api_invocation_time,datetime64[ns],0,109
is_deleted,boolean,0,1


# 2. Split data for training and testing

In [8]:
data = df[["record-identifier", "event-time", "total-est-attendance", "taxable-sales-usd"]].drop_duplicates()
prefix = "single-in-single-out-regression"

len(data)

2552

In [10]:
# 60 / 15 / 15 / 10 data split of temporal data (split by era rather than random)
end_train = len(data) * 60 // 100
end_test = len(data) * 75 // 100
end_validate = len(data) * 90 // 100

train_data = data.iloc[:end_train,:]
test_data = data.iloc[end_train: end_test,:]
validate_data = data.iloc[end_test: end_validate,:]
production_data = data.iloc[end_validate:,:]

len(train_data)

1531

In [ ]:
# create local stoage for split up data
model_data_path = Path(Path.cwd().parent, "data", "model", "single-regression")

train_file = model_data_path / "training_data.csv"
train_data.to_csv(train_file, index=False, header=False)

validation_file = model_data_path / "validation_data.csv"
test_data.to_csv(validation_file, index=False, header=False)

testing_file = model_data_path / "testing_data.csv"
validate_data.to_csv(testing_file, index=False, header=False)

production_file = model_data_path / "production_data.csv"
production_data.to_csv(production_file, index=False, header=False)

# ----- Optional -----
# s3_client = boto3.client("s3")

# Load the data into the s3 bucket as well.  Example call below:
# s3_client.upload_file(train_file, f"s3://{BUCKET}/{PROJECT}/{prefix}/train", "training_data")